In [1]:
import pyscf
import pyscf.mcscf
import numpy as np

In [2]:
spin_state = 0
nsite = 10
nspin = nsite // 2 # Half-filling
t = 1.0
U = 4.0
atom_str = "\n".join(
        f"H  {x + 0.5:7.4f} 0.0000  0.0000"
        for x in range(-nsite//2, nsite//2)
    )
mol = pyscf.gto.M()
mol.atom = atom_str
mol.symmetry = True
mol.nelectron = 2 * nspin
mol.incore_anyway = True
mol.spin = spin_state
mol.build()

In [3]:
e_nuc = mol.energy_nuc()

In [4]:
mf = pyscf.scf.RHF(mol)

In [5]:
h1 = np.zeros((nsite,nsite))
for i in range(nsite-1):
    h1[i,i+1] = h1[i+1,i] = -t
h1[nsite-1,0] = h1[0,nsite-1] = -t  # PBC
hub = np.zeros((nsite,nsite,nsite,nsite))
for i in range(nsite):
    hub[i,i,i,i] = U

In [6]:
mf.get_hcore = lambda *args: h1
mf.get_ovlp = lambda *args: np.eye(nsite)
mf._eri = pyscf.ao2mo.restore(8, hub, nsite)

In [7]:
mf.kernel() - e_nuc

converged SCF energy = 7.26338849588227


Overwritten attributes  get_ovlp get_hcore  of <class 'pyscf.scf.hf_symm.SymAdaptedRHF'>


np.float64(-2.944271909999159)

In [8]:
cas = pyscf.mcscf.CASCI(mf, nsite, (nspin, nspin))
mo = cas.sort_mo(range(nsite), base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), nsite)
exact_energy = cas.run().e_tot

CASCI E = 4.37333777093412  E(CI) = -5.83432263494731  S^2 = 0.0000000


In [9]:
mycc = pyscf.cc.CCSD(mf)
#mycc.verbose=4
ccsd_energy = mycc.kernel()

E(CCSD) = 4.073774845246623  E_corr = -3.189613650635649


In [10]:
from itertools import combinations
def point_to_k_basis(mf):
    arr = mf.mo_energy

    tolerance = 1e-7

    # Make group in which elements have same value
    value_to_indices = {}
    for idx, val in enumerate(arr):
        added = False
        for key in list(value_to_indices.keys()):
            if np.isclose(val, key, atol=tolerance): 
                value_to_indices[key].append(idx)
                added = True
                break
        if not added:
            value_to_indices[val] = [idx]

    # Make combination
    combos = []
    for value, indices in value_to_indices.items():
        if len(indices) > 1: 
            for combo in combinations(indices, 2):
                combos.append(combo)

    U = np.array([[1/np.sqrt(2), 1/np.sqrt(2)],[1j/np.sqrt(2), -1j/np.sqrt(2)]])
    U_tot = np.eye(len(mf.mo_energy), dtype=complex)
    for combo in combos:
        U_tot[combo[0]:combo[1]+1,combo[0]:combo[1]+1] = U

    # Gauge fixing
    import cmath
    gauges = []
    for i in range(len(mf.mo_energy)):
        # Value on origin leg 1 in OU matrix
        val  = mf.mo_coeff.dot(U_tot)[0,i]
        theta = cmath.phase(val)
        gauges.append(cmath.exp(-1j * theta))

    U_tot_gauge = U_tot.copy()
    for i in range(len(mf.mo_energy)):
        U_tot_gauge[:,i] = U_tot_gauge[:,i] * gauges[i]

    return U_tot_gauge

In [11]:
U_tot_gauge = point_to_k_basis(mf)
hcore_k = U_tot_gauge.T.conj().dot(hcore).dot(U_tot_gauge)
eri_k   = np.einsum('pqrs,pi,qj,rk,sl->ijkl', eri,
        U_tot_gauge.conj(), U_tot_gauge,
        U_tot_gauge.conj(), U_tot_gauge, optimize=True)

imag_max = max(np.max(hcore_k.imag), np.max(eri_k.imag))
print(f"Discarded imaginary part is {imag_max}.")
hcore_k_real = hcore_k.real
eri_k_real = eri_k.real

Discarded imaginary part is 1.8779637476200977e-10.


In [12]:
from qiskit_addon_sqd.counts import generate_counts_uniform
from qiskit_addon_sqd.counts import counts_to_arrays
 
 
rng = np.random.default_rng(24)
counts = generate_counts_uniform(10_000, nsite * 2, rand_seed=rng)
 
# Convert counts into bitstring and probability arrays
bitstring_matrix_full, probs_arr_full = counts_to_arrays(counts)

In [13]:
spin_sq = 0
open_shell = False

In [17]:
pyscf.fci.selected_ci.kernel_fixed_space = pyscf.fci.selected_ci.kernel_fixed_space_nosym

AttributeError: module 'pyscf.fci.selected_ci' has no attribute 'kernel_fixed_space_nosym'

In [ ]:
import numpy as np
from qiskit_addon_sqd.configuration_recovery import recover_configurations
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.subsampling import postselect_and_subsample

rng = np.random.default_rng(12345)

# SQD options
iterations = 5

# Eigenstate solver options
n_batches = 3
samples_per_batch = 850
max_davidson_cycles = 200

# Self-consistent configuration recovery loop
e_hist = np.zeros((iterations, n_batches))  # energy history
s_hist = np.zeros((iterations, n_batches))  # spin history
occupancy_hist = []
avg_occupancy = None
for i in range(iterations):
    print(f"Starting configuration recovery iteration {i}")
    # On the first iteration, we have no orbital occupancy information from the
    # solver, so we just post-select from the full bitstring set based on hamming weight.
    if avg_occupancy is None:
        bs_mat_tmp = bitstring_matrix_full
        probs_arr_tmp = probs_arr_full

    # If we have average orbital occupancy information, we use it to refine the full set of noisy configurations
    else:
        bs_mat_tmp, probs_arr_tmp = recover_configurations(
            bitstring_matrix_full,
            probs_arr_full,
            avg_occupancy,
            nspin,
            nspin,
            rand_seed=rng,
        )

    # Throw out configurations with incorrect particle number in either the spin-up or spin-down systems
    batches = postselect_and_subsample(
        bs_mat_tmp,
        probs_arr_tmp,
        hamming_right=nspin,
        hamming_left=nspin,
        samples_per_batch=samples_per_batch,
        num_batches=n_batches,
        rand_seed=rng,
    )

    # Run eigenstate solvers in a loop. This loop should be parallelized for larger problems.
    e_tmp = np.zeros(n_batches)
    s_tmp = np.zeros(n_batches)
    occs_tmp = []
    coeffs = []
    for j in range(n_batches):
        #energy_sci, coeffs_sci, avg_occs, spin = mod_solve_fermion(
        energy_sci, coeffs_sci, avg_occs, spin = solve_fermion(
            batches[j],
            hcore_k_real,
            eri_k_real,
            #hcore,
            #eri,
            open_shell=open_shell,
            spin_sq=spin_sq,
            max_davidson=max_davidson_cycles,
#            nosym = True
        )
        energy_sci += nuclear_repulsion_energy
        e_tmp[j] = energy_sci
        s_tmp[j] = spin
        occs_tmp.append(avg_occs)
        coeffs.append(coeffs_sci)

    # Combine batch results
    avg_occupancy = np.mean(occs_tmp, axis=0)

    # Track optimization history
    e_hist[i, :] = e_tmp
    s_hist[i, :] = s_tmp
    occupancy_hist.append(avg_occupancy)

Starting configuration recovery iteration 0
Starting configuration recovery iteration 1


In [ ]:
import matplotlib.pyplot as plt

# Data for energies plot
x1 = range(iterations)
e_diff = [abs(np.min(energies) - exact_energy) for energies in e_hist]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = occupancy_hist[-1][0] + occupancy_hist[-1][1]
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(y=chem_accuracy, color="#BF5700", linestyle="--", label="chemical accuracy")
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

In [ ]:
e_diff